In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/sample_submission.csv
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/train.parquet
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/test.parquet
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/Data_schema/DER/MEDA_RHS_derived_data_schema.csv
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/Data_schema/DER/MEDA_ANCILLARY_derived_data_schema.csv
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/Data_schema/DER/MEDA_PS_derived_data_schema.csv
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/Data_schema/DER/MEDA_TIRS_derived_data_schema.csv
/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/Docs/

In [2]:
import polars as pl


In [3]:
# Defining the file path
train_path = "/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/train.parquet"
test_path = "/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/test.parquet"

In [4]:
# 2. CREATE THE LAZY FRAMES
# This reads the metadata but keeps the heavy data on the hard drive
print("Setting up LazyFrames...")
train_lazy = pl.scan_parquet(train_path)
test_lazy = pl.scan_parquet(test_path)


Setting up LazyFrames...


In [5]:
print("=== FIRST 5 ROWS OF TRAIN DATA ===")
# We ask for the head(5), and then trigger it with collect()
train_head = train_lazy.head(5).collect()
print(train_head)

print("\n=== FIRST 5 ROWS OF TEST DATA ===")
test_head = test_lazy.head(5).collect()
print(test_head)

=== FIRST 5 ROWS OF TRAIN DATA ===
shape: (5, 36)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ SCLK      ┆ LMST      ┆ LTST      ┆ SOLAR_LON ┆ … ┆ TIRS_GROU ┆ ROVER_HGA ┆ SKYCAM_OF ┆ ROVER_ST │
│ ---       ┆ ---       ┆ ---       ┆ GITUDE_AN ┆   ┆ ND_FOOTPR ┆ _OFF      ┆ F         ┆ ILL      │
│ i64       ┆ str       ┆ str       ┆ GLE       ┆   ┆ INT_NOT_I ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆ ---       ┆   ┆ N_S…      ┆ f64       ┆ f64       ┆ f64      │
│           ┆           ┆           ┆ f64       ┆   ┆ ---       ┆           ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 667042464 ┆ 00001M16: ┆ 0001      ┆ 101.03    ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│           ┆ 05:31.315 ┆ 15:28:01  ┆    

In [6]:
# --- EDA 1: THE STRUCTURAL HANDSHAKE ---
print("\n=== 1. COLUMN CHECK ===")
train_cols = set(train_lazy.columns)
test_cols = set(test_lazy.columns)
print(f"Columns in Train but NOT in Test: {train_cols - test_cols}")
print(f"Columns in Test but NOT in Train: {test_cols - train_cols}")


=== 1. COLUMN CHECK ===
Columns in Train but NOT in Test: {'PRESSURE'}
Columns in Test but NOT in Train: set()


/tmp/ipykernel_17/3356962114.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  train_cols = set(train_lazy.columns)
/tmp/ipykernel_17/3356962114.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  test_cols = set(test_lazy.columns)


In [7]:
print("\n=== 2. TIMELINE CHECK (SCLK) ===")

# We ask Polars to find the earliest (min) and latest (max) time in the Train set
train_time = train_lazy.select([
    pl.col("SCLK").min().alias("start_time"),
    pl.col("SCLK").max().alias("end_time")
]).collect()

# We do the exact same thing for the Test set
test_time = train_lazy.select([
    pl.col("SCLK").min().alias("start_time"),
    pl.col("SCLK").max().alias("end_time")
]).collect()

# Print out the actual numbers (the [0] just extracts the number from the Polars table)
print(f"Train Timeline: Starts at {train_time['start_time'][0]} | Ends at {train_time['end_time'][0]}")
print(f"Test Timeline:  Starts at {test_time['start_time'][0]}  | Ends at {test_time['end_time'][0]}")


=== 2. TIMELINE CHECK (SCLK) ===
Train Timeline: Starts at 667042464 | Ends at 675857016
Test Timeline:  Starts at 667042464  | Ends at 675857016


In [8]:
print("=== 3. THE MISSING DATA HUNT (TRAIN SET) ===")

# .null_count() scans every column and adds up the blank spaces
# .collect() executes the scan and brings the result to your screen
missing_values = train_lazy.null_count().collect()

# We transpose it (.transpose()) just to make it easier to read top-to-bottom
# include_header_name=True keeps the sensor names attached to the numbers
print(missing_values.transpose(include_header=True, header_name="Sensor_Name", column_names=["Missing_Count"]))

=== 3. THE MISSING DATA HUNT (TRAIN SET) ===
shape: (36, 2)
┌─────────────────────────────────┬───────────────┐
│ Sensor_Name                     ┆ Missing_Count │
│ ---                             ┆ ---           │
│ str                             ┆ u32           │
╞═════════════════════════════════╪═══════════════╡
│ SCLK                            ┆ 0             │
│ LMST                            ┆ 0             │
│ LTST                            ┆ 0             │
│ SOLAR_LONGITUDE_ANGLE           ┆ 0             │
│ SOLAR_ZENITHAL_ANGLE            ┆ 0             │
│ …                               ┆ …             │
│ ROVER_LOW_TILT                  ┆ 130673        │
│ TIRS_GROUND_FOOTPRINT_NOT_IN_S… ┆ 130673        │
│ ROVER_HGA_OFF                   ┆ 8130533       │
│ SKYCAM_OFF                      ┆ 8130533       │
│ ROVER_STILL                     ┆ 130673        │
└─────────────────────────────────┴───────────────┘


In [9]:
print("=== 4. PRESSURE HEALTH CHECK (TRAIN SET) ===")

# We strictly isolate the PRESSURE column, pull it into memory, and calculate stats
pressure_stats = train_lazy.select("PRESSURE").collect().describe()

print(pressure_stats)

=== 4. PRESSURE HEALTH CHECK (TRAIN SET) ===
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ PRESSURE   │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 8.130533e6 │
│ null_count ┆ 0.0        │
│ mean       ┆ 749.645676 │
│ std        ┆ 8.923145   │
│ min        ┆ 713.91     │
│ 25%        ┆ 744.46     │
│ 50%        ┆ 750.22     │
│ 75%        ┆ 755.94     │
│ max        ┆ 771.44     │
└────────────┴────────────┘


In [10]:
#EDA IS COMPLETED FOR THE INTIAL PHASE
#

The Evidence: The Train and Test sets have the exact same start and end times (SCLK).

The Next Step: Because we are filling holes in the middle of a timeline rather than predicting the unknown future, we are allowed to use a powerful technique called Forward-Filling and Backward-Filling. If a sensor dies for 5 minutes, we can just look at the minute before it died and the minute after it turned back on to mathematically estimate the missing numbers.

The Evidence: Sensors like ROVER_HGA_OFF and SKYCAM_OFF are missing over 8.1 million rows (basically 99% of the data).

The Next Step (Action: Drop): We must permanently delete these columns from both the Train and Test datasets. If you feed a column that is 99% blank into a machine learning model, it acts like noise and destroys the model's accuracy.

The Evidence: Sensors like ROVER_STILL and ROVER_LOW_TILT are missing about 130,000 rows (roughly 1.5% of the data).

The Next Step (Action: Impute): We cannot drop these because 98.5% of the data is perfectly good! Instead, we will write a Polars script to "impute" (patch) these holes.  For a rover parked on Mars, if ROVER_STILL was "True" at 1:00 PM, and the sensor glitched at 1:01 PM, it is statistically safe to assume it was still parked at 1:01 PM.

The Evidence: The LMST (Local Mean Solar Time) column is stored as text (e.g., "14:30:00").

The Next Step (Action: Feature Engineering): Algorithms only understand math, not text. Furthermore, time is cyclical (23:59 wraps back around to 00:00). We will need to use trigonometric functions (sine and cosine) to translate that text into a mathematical circle so the model understands the repeating rhythm of the Martian day. This is a classic, highly effective technique in astrostatistics.

In [11]:
#2: Data Cleaning.
print("=== PHASE 2: DATA CLEANING START ===")

# 1. Define the Black Holes we want to destroy
columns_to_drop = ["ROVER_HGA_OFF", "SKYCAM_OFF"]

# 2. Build the cleaning pipeline
# Notice how we chain commands together inside the parentheses. 
# This is the beauty of Polars LazyFrames!
clean_train_lazy = (
    train_lazy
    .drop(columns_to_drop)              # Action 1: Delete the 99% empty columns
    .sort("SCLK")                       # Action 2: Put everything in chronological order
    .fill_null(strategy="forward")      # Action 3: Drag the last known value forward to fill holes
    .fill_null(strategy="backward")     # Action 4: Drag the next value backward (just in case the very first row is a hole)
)

print("Pipeline built successfully!")

# 3. Verify the Cleaning Worked
print("\n=== VERIFYING THE FIX ===")
# We check our old "Swiss Cheese" columns to prove they are fixed
verification = clean_train_lazy.select(
    pl.col("ROVER_LOW_TILT").null_count().alias("Tilt_Missing"),
    pl.col("ROVER_STILL").null_count().alias("Still_Missing")
).collect()

print(verification)

=== PHASE 2: DATA CLEANING START ===
Pipeline built successfully!

=== VERIFYING THE FIX ===
shape: (1, 2)
┌──────────────┬───────────────┐
│ Tilt_Missing ┆ Still_Missing │
│ ---          ┆ ---           │
│ u32          ┆ u32           │
╞══════════════╪═══════════════╡
│ 0            ┆ 0             │
└──────────────┴───────────────┘


In [12]:
print("=== APPLYING PIPELINE TO TEST SET ===")
# We use the exact same columns_to_drop list from earlier
clean_test_lazy = (
    test_lazy
    .drop(columns_to_drop)
    .sort("SCLK")
    .fill_null(strategy="forward")
    .fill_null(strategy="backward")
)
print("Test set successfully cleaned!")

print("\n=== PREPARING FOR FEATURE ENGINEERING ===")
print("Looking at the raw Local Mean Solar Time (LMST):")
# Let's look at the first 5 rows of the time column
lmst_sample = clean_train_lazy.select("LMST").head(5).collect()
print(lmst_sample)

=== APPLYING PIPELINE TO TEST SET ===
Test set successfully cleaned!

=== PREPARING FOR FEATURE ENGINEERING ===
Looking at the raw Local Mean Solar Time (LMST):
shape: (5, 1)
┌────────────────────┐
│ LMST               │
│ ---                │
│ str                │
╞════════════════════╡
│ 00001M16:05:31.315 │
│ 00001M16:05:32.289 │
│ 00001M16:05:33.262 │
│ 00001M16:05:34.235 │
│ 00001M16:05:35.208 │
└────────────────────┘


In [13]:
print("=== DECODING THE NASA TIME FORMAT ===")

# We use .with_columns() to create brand new columns in our dataset without deleting the old ones
clean_train_lazy = clean_train_lazy.with_columns([
    
    # 1. Slice the first 5 characters and turn them into an integer (The Sol)
    pl.col("LMST").str.slice(0, 5).cast(pl.Int32).alias("Sol"),
    
    # 2. Slice the hours, minutes, and seconds, and do the math to make a Decimal Hour
    (
        pl.col("LMST").str.slice(6, 2).cast(pl.Float64) +                  # The Hour
        (pl.col("LMST").str.slice(9, 2).cast(pl.Float64) / 60.0) +         # The Minutes / 60
        (pl.col("LMST").str.slice(12).cast(pl.Float64) / 3600.0)           # The Seconds / 3600
    ).alias("LMST_Decimal")
])

# Let's peek at our newly created columns!
decoded_time = clean_train_lazy.select(["LMST", "Sol", "LMST_Decimal"]).head(5).collect()
print(decoded_time)

=== DECODING THE NASA TIME FORMAT ===
shape: (5, 3)
┌────────────────────┬─────┬──────────────┐
│ LMST               ┆ Sol ┆ LMST_Decimal │
│ ---                ┆ --- ┆ ---          │
│ str                ┆ i32 ┆ f64          │
╞════════════════════╪═════╪══════════════╡
│ 00001M16:05:31.315 ┆ 1   ┆ 16.092032    │
│ 00001M16:05:32.289 ┆ 1   ┆ 16.092302    │
│ 00001M16:05:33.262 ┆ 1   ┆ 16.092573    │
│ 00001M16:05:34.235 ┆ 1   ┆ 16.092843    │
│ 00001M16:05:35.208 ┆ 1   ┆ 16.093113    │
└────────────────────┴─────┴──────────────┘


In [14]:
import math

print("=== 5. BENDING TIME INTO A CIRCLE ===")

# 1. Apply Sine and Cosine to the Train set
clean_train_lazy = clean_train_lazy.with_columns([
    (pl.col("LMST_Decimal") * (2 * math.pi / 24.0)).sin().alias("LMST_sin"),
    (pl.col("LMST_Decimal") * (2 * math.pi / 24.0)).cos().alias("LMST_cos")
])

# 2. Apply the string-slicing AND the Sine/Cosine to the Test set
clean_test_lazy = clean_test_lazy.with_columns([
    pl.col("LMST").str.slice(0, 5).cast(pl.Int32).alias("Sol"),
    (
        pl.col("LMST").str.slice(6, 2).cast(pl.Float64) + 
        (pl.col("LMST").str.slice(9, 2).cast(pl.Float64) / 60.0) + 
        (pl.col("LMST").str.slice(12).cast(pl.Float64) / 3600.0)
    ).alias("LMST_Decimal")
]).with_columns([
    (pl.col("LMST_Decimal") * (2 * math.pi / 24.0)).sin().alias("LMST_sin"),
    (pl.col("LMST_Decimal") * (2 * math.pi / 24.0)).cos().alias("LMST_cos")
])

print("Time successfully bent into a circle for both datasets!")

# 3. Verify the final math on the Train set
final_time_check = clean_train_lazy.select(["LMST_Decimal", "LMST_sin", "LMST_cos"]).head(5).collect()
print(final_time_check)

=== 5. BENDING TIME INTO A CIRCLE ===
Time successfully bent into a circle for both datasets!
shape: (5, 3)
┌──────────────┬───────────┬───────────┐
│ LMST_Decimal ┆ LMST_sin  ┆ LMST_cos  │
│ ---          ┆ ---       ┆ ---       │
│ f64          ┆ f64       ┆ f64       │
╞══════════════╪═══════════╪═══════════╡
│ 16.092032    ┆ -0.87782  ┆ -0.478991 │
│ 16.092302    ┆ -0.877854 ┆ -0.478929 │
│ 16.092573    ┆ -0.877888 ┆ -0.478867 │
│ 16.092843    ┆ -0.877922 ┆ -0.478805 │
│ 16.093113    ┆ -0.877955 ┆ -0.478742 │
└──────────────┴───────────┴───────────┘


In [15]:
print("=== PHASE 3: PREPARING DATA FOR XGBOOST ===")

# 1. Collect the final cleaned data into memory (this might take a few seconds for 8M rows)
# We convert it to Pandas because Scikit-Learn and XGBoost prefer Pandas/Numpy arrays
final_train_df = clean_train_lazy.collect().to_pandas()
final_test_df = clean_test_lazy.collect().to_pandas()

# 2. Separate the Clues (X) from the Answer (y) in the Training Set
# We drop PRESSURE to create our clues, and we isolate PRESSURE to create our target
X_train = final_train_df.drop(columns=["PRESSURE"])
y_train = final_train_df["PRESSURE"]

# 3. The Test set doesn't have a PRESSURE column, so it is already purely our X_test
X_test = final_test_df

print(f"X_train shape (Clues): {X_train.shape}")
print(f"y_train shape (Answers): {y_train.shape}")
print(f"X_test shape (Final Exam): {X_test.shape}")

=== PHASE 3: PREPARING DATA FOR XGBOOST ===
X_train shape (Clues): (8130533, 37)
y_train shape (Answers): (8130533,)
X_test shape (Final Exam): (3949990, 37)


In [16]:
print("=== EMERGENCY SURGERY: DROPPING STRINGS ===")

# Forcefully drop the string columns from our training clues (X_train)
X_train = X_train.drop(columns=["LMST", "LTST"])

# Forcefully drop the string columns from our final exam (X_test)
X_test = X_test.drop(columns=["LMST", "LTST"])

print("String columns completely destroyed.")

# Verify the shapes. The number of Clues should have dropped from 37 to 35.
print(f"X_train new shape: {X_train.shape}")
print(f"X_test new shape: {X_test.shape}")

=== EMERGENCY SURGERY: DROPPING STRINGS ===
String columns completely destroyed.
X_train new shape: (8130533, 35)
X_test new shape: (3949990, 35)


In [17]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

print("=== PHASE 4: TRAINING THE XGBOOST MODEL ===")

# 1. Initialize the Model
# We are telling it to build 100 mistake-fixing trees (n_estimators)
model = xgb.XGBRegressor(
    n_estimators=100,      
    learning_rate=0.1,     
    tree_method="hist",    # The secret to fast training on 8M rows!
    random_state=42        # Ensures we get the exact same result every time
)

# 2. Train the Model (The "Fit" step)
print("Training in progress... (Grab a sip of water, this takes a minute!)")
model.fit(X_train, y_train)
print("Model Training Complete! 🚀")

# 3. The Sanity Check: How well did it learn?
# We ask it to predict the training data it just studied, and check its grade
train_predictions = model.predict(X_train)
train_mse = mean_squared_error(y_train, train_predictions)

print(f"Training Mean Squared Error (MSE): {train_mse:.4f}")

=== PHASE 4: TRAINING THE XGBOOST MODEL ===
Training in progress... (Grab a sip of water, this takes a minute!)
Model Training Complete! 🚀
Training Mean Squared Error (MSE): 0.1742


In [18]:
import pandas as pd

print("=== FINAL STEP: GENERATING KAGGLE SUBMISSION CSV ===")

# 1. Ask your XGBoost model to generate the real predictions
final_predictions = model.predict(X_test)

# 2. PASTE YOUR COPIED PATH IN THE QUOTES BELOW!
submission_path = '/kaggle/input/competitions/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/sample_submission.csv' 
submission = pd.read_csv(submission_path)

# 3. Overwrite Kaggle's dummy numbers with your accurate predictions
submission['PRESSURE'] = final_predictions

# 4. Export the final dataframe to a new CSV file
submission.to_csv('submission.csv', index=False)

print("Success! 'submission.csv' has been generated.")
print("\n=== VERIFYING FINAL FORMAT ===")
print(submission.head())

=== FINAL STEP: GENERATING KAGGLE SUBMISSION CSV ===
Success! 'submission.csv' has been generated.

=== VERIFYING FINAL FORMAT ===
      row_id    PRESSURE
0  684737856  760.674866
1  684737857  760.760132
2  684737858  760.760132
3  684737859  760.760132
4  684737860  760.760132
